# Use Case 8: SQL Engine on Embedded KV Store

**The Concept:** 
Sometimes you need the simplicity of SQL queries without spinning up a full database server. SochDB includes a built-in **SQL engine** that executes queries directly on its embedded KV store — no external dependencies.

**The Architecture:** 
The `SQLExecutor` translates standard SQL (CREATE TABLE, INSERT, SELECT, UPDATE, DELETE) into KV operations. It supports WHERE clauses, ORDER BY, LIMIT/OFFSET, COUNT, and LIKE patterns. Tables are stored as KV entries with automatic schema management.

---

### Step 0: Install Packages

In [1]:
!pip install sochdb

You should consider upgrading via the '/Users/sushanth/sochdb_python/venv/bin/python3 -m pip install --upgrade pip' command.


### Step 1: Initialize Database
Open an embedded database — no servers, no config files.

In [2]:
from sochdb import Database

db = Database.open("./sql_engine_demo_db")
print("Database opened for SQL Engine demo.")

Database opened for SQL Engine demo.


### Step 2: Create Tables
Define tables with typed columns: `INT`, `TEXT`, `FLOAT`, `BOOL`.

In [3]:
# Create a products table
db.execute("CREATE TABLE products (id INT, name TEXT, category TEXT, price FLOAT, in_stock BOOL)")
print("Created 'products' table.")

# Create an orders table
db.execute("CREATE TABLE orders (id INT, product_id INT, customer TEXT, quantity INT, total FLOAT)")
print("Created 'orders' table.")

# List all tables
tables = db.list_tables()
print(f"\nAll tables: {tables}")

Created 'products' table.
Created 'orders' table.

All tables: ['orders', 'products']


### Step 3: Insert Data
Populate the tables with sample data.

In [5]:
# Insert products
products = [
    (1, "SochDB Enterprise", "Database", 999.99, True),
    (2, "VectorSearch Pro", "AI Tool", 499.50, True),
    (3, "DataPipeline SDK", "ETL", 299.00, False),
    (4, "GraphDB Lite", "Database", 149.99, True),
    (5, "MLOps Platform", "AI Tool", 799.00, True),
    (6, "StreamProcessor", "ETL", 399.00, True),
    (7, "CacheLayer Ultra", "Database", 599.99, False),
]

for p in products:
    db.execute(f"INSERT INTO products (id, name, category, price, in_stock) VALUES ({p[0]}, '{p[1]}', '{p[2]}', {p[3]}, {str(p[4]).lower()})")

print(f"Inserted {len(products)} products.")

# Insert orders
orders = [
    (101, 1, "Acme Corp", 5, 4999.95),
    (102, 2, "TechStart Inc", 10, 4995.00),
    (103, 1, "BigData Ltd", 3, 2999.97),
    (104, 5, "AI Solutions", 2, 1598.00),
    (105, 4, "DevHouse", 20, 2999.80),
    (106, 6, "DataFlow Co", 7, 2793.00),
]

for o in orders:
    db.execute(f"INSERT INTO orders (id, product_id, customer, quantity, total) VALUES ({o[0]}, {o[1]}, '{o[2]}', {o[3]}, {o[4]})")

print(f"Inserted {len(orders)} orders.")

Inserted 7 products.
Inserted 6 orders.


### Step 4: Basic SELECT Queries
Query data with standard SQL syntax.

In [6]:
# Select all products
result = db.execute("SELECT * FROM products")
print("All Products:")
print(f"Columns: {result.columns}")
for row in result.rows:
    print(f"  {row}")

All Products:
Columns: ['id', 'name', 'category', 'price', 'in_stock']
  {'id': 1, 'name': 'SochDB Enterprise', 'category': 'Database', 'price': 999.99, 'in_stock': True}
  {'id': 2, 'name': 'VectorSearch Pro', 'category': 'AI Tool', 'price': 499.5, 'in_stock': True}
  {'id': 3, 'name': 'DataPipeline SDK', 'category': 'ETL', 'price': 299.0, 'in_stock': False}
  {'id': 4, 'name': 'GraphDB Lite', 'category': 'Database', 'price': 149.99, 'in_stock': True}
  {'id': 5, 'name': 'MLOps Platform', 'category': 'AI Tool', 'price': 799.0, 'in_stock': True}
  {'id': 6, 'name': 'StreamProcessor', 'category': 'ETL', 'price': 399.0, 'in_stock': True}
  {'id': 7, 'name': 'CacheLayer Ultra', 'category': 'Database', 'price': 599.99, 'in_stock': False}


### Step 5: WHERE Clauses & Filtering
Filter results using comparison operators and boolean conditions.

In [7]:
# Products in the "Database" category that are in stock
result = db.execute("SELECT name, price FROM products WHERE category = 'Database' AND in_stock = true")
print("In-stock Database products:")
for row in result.rows:
    print(f"  {row}")

print()

# Products cheaper than $500
result = db.execute("SELECT name, price FROM products WHERE price < 500")
print("Products under $500:")
for row in result.rows:
    print(f"  {row}")

In-stock Database products:
  {'name': 'SochDB Enterprise', 'price': 999.99}
  {'name': 'GraphDB Lite', 'price': 149.99}

Products under $500:
  {'name': 'VectorSearch Pro', 'price': 499.5}
  {'name': 'DataPipeline SDK', 'price': 299.0}
  {'name': 'GraphDB Lite', 'price': 149.99}
  {'name': 'StreamProcessor', 'price': 399.0}


### Step 6: LIKE Pattern Matching
Use SQL LIKE for text pattern matching.

In [8]:
# Products with names containing "DB"
result = db.execute("SELECT name, category FROM products WHERE name LIKE '%DB%'")
print("Products with 'DB' in name:")
for row in result.rows:
    print(f"  {row}")

print()

# Products starting with 'S'
result = db.execute("SELECT name FROM products WHERE name LIKE 'S%'")
print("Products starting with 'S':")
for row in result.rows:
    print(f"  {row}")

Products with 'DB' in name:
  {'name': 'SochDB Enterprise', 'category': 'Database'}
  {'name': 'GraphDB Lite', 'category': 'Database'}

Products starting with 'S':
  {'name': 'SochDB Enterprise'}
  {'name': 'StreamProcessor'}


### Step 7: ORDER BY, LIMIT & OFFSET
Sort and paginate results.

In [9]:
# Top 3 most expensive products
result = db.execute("SELECT name, price FROM products ORDER BY price LIMIT 3")
print("Top 3 cheapest products:")
for row in result.rows:
    print(f"  {row}")

print()

# Page 2 of products (skip first 3, show next 3)
result = db.execute("SELECT name, price FROM products ORDER BY price LIMIT 3 OFFSET 3")
print("Products page 2 (offset 3, limit 3):")
for row in result.rows:
    print(f"  {row}")

Top 3 cheapest products:
  {'name': 'GraphDB Lite', 'price': 149.99}
  {'name': 'DataPipeline SDK', 'price': 299.0}
  {'name': 'StreamProcessor', 'price': 399.0}

Products page 2 (offset 3, limit 3):
  {'name': 'VectorSearch Pro', 'price': 499.5}
  {'name': 'CacheLayer Ultra', 'price': 599.99}
  {'name': 'MLOps Platform', 'price': 799.0}


### Step 8: COUNT Aggregation
Count rows matching conditions.

In [10]:
# Total number of products
result = db.execute("SELECT COUNT(*) FROM products")
print(f"Total products: {result.rows[0]}")

# Number of in-stock products
result = db.execute("SELECT COUNT(*) FROM products WHERE in_stock = true")
print(f"In-stock products: {result.rows[0]}")

# Number of orders
result = db.execute("SELECT COUNT(*) FROM orders")
print(f"Total orders: {result.rows[0]}")

Total products: {'count': 7}
In-stock products: {'count': 5}
Total orders: {'count': 6}


### Step 9: UPDATE Records
Modify existing data.

In [11]:
# Put DataPipeline SDK back in stock
result = db.execute("UPDATE products SET in_stock = true WHERE id = 3")
print(f"Updated {result.rows_affected} row(s).")

# Give a 10% discount to all AI Tools (update price)
result = db.execute("UPDATE products SET price = 449.55 WHERE id = 2")
print(f"Updated {result.rows_affected} row(s).")

# Verify
result = db.execute("SELECT name, price, in_stock FROM products WHERE id = 2 OR id = 3")
print("\nUpdated products:")
for row in result.rows:
    print(f"  {row}")

Updated 1 row(s).
Updated 1 row(s).

Updated products:


### Step 10: DELETE Records
Remove data from tables.

In [13]:
# Delete out-of-stock products
result = db.execute("DELETE FROM products WHERE in_stock = false")
print(f"Deleted {result.rows_affected} out-of-stock product(s).")

# Verify remaining products
result = db.execute("SELECT name, in_stock FROM products")
print(f"\nRemaining products ({len(result.rows)}):")
for row in result.rows:
    print(f"  {row}")

Deleted 1 out-of-stock product(s).

Remaining products (6):
  {'name': 'SochDB Enterprise', 'in_stock': True}
  {'name': 'VectorSearch Pro', 'in_stock': True}
  {'name': 'DataPipeline SDK', 'in_stock': True}
  {'name': 'GraphDB Lite', 'in_stock': True}
  {'name': 'MLOps Platform', 'in_stock': True}
  {'name': 'StreamProcessor', 'in_stock': True}


### Step 11: Table Schema Inspection
Inspect the schema of any table.

In [14]:
# Get schema for the products table
schema = db.get_table_schema("products")
print(f"Products table schema: {schema}")

schema = db.get_table_schema("orders")
print(f"Orders table schema: {schema}")

# List all tables in the database
print(f"\nAll tables: {db.list_tables()}")

Products table schema: {'table': 'products', 'columns': [{'name': 'id', 'type': 'INT', 'primary_key': False}, {'name': 'name', 'type': 'TEXT', 'primary_key': False}, {'name': 'category', 'type': 'TEXT', 'primary_key': False}, {'name': 'price', 'type': 'FLOAT', 'primary_key': False}, {'name': 'in_stock', 'type': 'BOOL', 'primary_key': False}], 'primary_key': None}
Orders table schema: {'table': 'orders', 'columns': [{'name': 'id', 'type': 'INT', 'primary_key': False}, {'name': 'product_id', 'type': 'INT', 'primary_key': False}, {'name': 'customer', 'type': 'TEXT', 'primary_key': False}, {'name': 'quantity', 'type': 'INT', 'primary_key': False}, {'name': 'total', 'type': 'FLOAT', 'primary_key': False}], 'primary_key': None}

All tables: ['orders', 'products']


### Cleanup

In [15]:
# Drop tables
db.execute("DROP TABLE products")
db.execute("DROP TABLE orders")
print(f"Tables after cleanup: {db.list_tables()}")

db.close()
print("Database closed.")

Tables after cleanup: []
Database closed.
